In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.cuda.amp import GradScaler, autocast
import torch.cuda as cuda
import time
import uuid
from prune_neurals import Prunner
import copy


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

batch_size = 128


# Data augmentation và chuẩn hóa cho CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load dữ liệu CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
# Hàm test
def test(model_):
    model_.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = model_(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f'Accuracy on test set: {accuracy:.2f}%')
    return accuracy



Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda
Using device: cuda


100%|██████████| 170M/170M [00:04<00:00, 41.9MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
a = Prunner().method_name
a

{'Prune neurels': ['base', 'kmeans', 'distance-based-clustering', 'kmedoids'],
 'Prune dataset': ['base', 'kmeans', 'distance-based-clustering', 'kmedoids']}

In [ ]:

if __name__ == '__main__':
    import time
    import numpy as np

    start = time.time()
    # Load mô hình VGG16 với weights pretrained
    model = models.vgg16(weights=False)

    # Thay đổi lớp fully connected cuối để phù hợp với CIFAR-10 (10 classes)
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, 10)
    model.to('cuda')
    model.load_state_dict(torch.load("/prune_neurals/best_model_vgg16_cuda.pth", weights_only=False))

    print("load successfully")

    # Debug: Print classifier structure before pruning
    print("Classifier structure before pruning:")
    for i, layer in enumerate(model.classifier):
        if hasattr(layer, 'weight'):
            print(f"Layer {i}: {type(layer).__name__} - in_features: {layer.in_features}, out_features: {layer.out_features}")
        else:
            print(f"Layer {i}: {type(layer).__name__}")

    res1 = {}
    pruner = Prunner()
    # test in prunner 1 layer
    print('test in prunner 1 layer')
    ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    for method in pruner.method_name['Prune neurels'][0:1]:
      method_dict = {}
      for ratio in ratios:
        acc = []
        acc2 = []

        for i in range(3):
          test_model1 = copy.deepcopy(model)
          test_model2 = copy.deepcopy(model)
          new_layer_1, new_layer_2 = pruner.prune_neurals(test_model1.classifier[0], test_model1.classifier[3], prune_ratio=ratio, method=method, device='cuda')
          test_model1.classifier[0] = new_layer_1
          test_model1.classifier[3] = new_layer_2
          # print(f'test {method} in ratio {ratio} {i} time:')
          # test(test_model1)
          acc.append(test(test_model1))
          new_layer_2, new_layer_3 = pruner.prune_neurals(test_model2.classifier[3], test_model2.classifier[6], prune_ratio=ratio, method=method, device='cuda')
          test_model2.classifier[3] = new_layer_2
          test_model2.classifier[6] = new_layer_3
          # print(f'test {method} in ratio {ratio} {i} time:')
          # test(test_model2)
          acc2.append(test(test_model2))

        mean_acc = (acc[0]+ acc[1] + acc[2])/3
        mean_acc2 = (acc2[0] + acc2[1] + acc2[2])/3
        method_dict[ratio] = {1:mean_acc, 2: mean_acc2}
        # bre?ak
      res1[method] = method_dict
      # break


    res2 = {}
    #test in prunner 2 layer
    print('test in prunner 2 layer')
    ratios_2layer = [0.8, 0.9]
    for method in pruner.method_name['Prune neurels'][0:1]:
      method_dict = {}
      for ratio in ratios_2layer:
        acc = []
        for i in range(3):
          test_model = copy.deepcopy(model)
          new_layer_1, new_layer_2 = pruner.prune_neurals(test_model.classifier[0], test_model.classifier[3], prune_ratio=ratio, method=method, device='cuda')
          new_layer_2, new_layer_3 = pruner.prune_neurals(new_layer_2, test_model.classifier[6], prune_ratio=ratio, method=method, device='cuda')
          test_model.classifier[0] = new_layer_1
          test_model.classifier[3] = new_layer_2
          test_model.classifier[6] = new_layer_3
          # print(f'test {method} in ratio {ratio} {i} time:')
          # test(test_model)
          acc.append(test(test_model))
        mean_acc = (acc[0]+ acc[1] + acc[2])/3
        method_dict[ratio] = mean_acc
      res2[method] = method_dict

RuntimeError: PytorchStreamReader failed reading zip archive: invalid header or archive is corrupted

In [ ]:
res1

{'kmeans': {0.1: {1: 91.34666666666665, 2: 91.25999999999999},
  0.2: {1: 91.29666666666667, 2: 91.31333333333333},
  0.3: {1: 91.29, 2: 91.25},
  0.4: {1: 91.33999999999999, 2: 91.32666666666667},
  0.5: {1: 91.23, 2: 91.26333333333334},
  0.6: {1: 91.02999999999999, 2: 91.24333333333334},
  0.7: {1: 91.03333333333335, 2: 91.25666666666666},
  0.8: {1: 90.96333333333332, 2: 91.01666666666667},
  0.9: {1: 90.58666666666666, 2: 90.66333333333334}}}

In [ ]:
res2

{'kmeans': {0.8: 90.17666666666668, 0.9: 83.76666666666667}}

In [ ]:
import json

with open('out22.json', 'w') as f:
    json.dump(res2, f)